In [ ]:
import pickle
import pandas as pd
import numpy as np
from src.utils import load_env, set_seed, get_logger
from case_study.utils import set_matplot_style
from src.experiment_config import ExperimentConfig
from src.data_loading import DatasetLoader


set_matplot_style()
env_vars = load_env()
logger = get_logger("cluster_lr_eval")
set_seed(env_vars["RANDOM_SEED"])

In [ ]:
DATASET = "tweets_immigration"
LABELED_ONLY = True
CLUSTER_TEC = "kmeans"
PCK_WEIGHT = 0.1
K = 200
EMB_NAME = "mets_only_ling_mets" # "mets_only_qwen_fe"

config = ExperimentConfig(
        task_type="cluster_lr",
        task_name="",
        dataset=DATASET,
        labeled_only=LABELED_ONLY,
        logger=logger,
        env_vars=env_vars
)

In [ ]:
# load metaphor embedding data
met_emb_data_path = f"{env_vars["RESULTS_DIR"]}/gen_embeddings/sbert/{config.dataset_out_name}_{EMB_NAME}_embeddings.pkl"
with open(met_emb_data_path, "rb") as f:
    met_emb_data = pickle.load(f)

In [ ]:
met_clust_data_path = f"{env_vars["RESULTS_DIR"]}/cluster_embeddings/{CLUSTER_TEC}/{config.dataset_out_name}/{EMB_NAME}_clusters_{K}.pkl"
if CLUSTER_TEC == "pckmeans":
    met_clust_data_path = met_clust_data_path.replace(".pkl", f"_w{PCK_WEIGHT}.pkl")

with open(met_clust_data_path, "rb") as f:
    met_clust_data = pickle.load(f)

In [ ]:
# read in model weights
# concept_weight_path = f"{env_vars["RESULTS_DIR"]}/cluster_lr/Hero/combined_clusters/k_t75_m50_weights.pkl"
polarity_weights = f"{env_vars['RESULTS_DIR']}/cluster_lr/{config.dataset_out_name}/polarity/{CLUSTER_TEC}/{EMB_NAME}/k_{K}_weights.pkl"
if CLUSTER_TEC == "pckmeans":
    polarity_weights = polarity_weights.replace("_weights.pkl", f"_w{PCK_WEIGHT}_weights.pkl")
with open(polarity_weights, "rb") as f:
    concept_weights = pickle.load(f)

In [ ]:
print(type(concept_weights))
print(concept_weights.shape)

In [ ]:
# Mean absolute weight across all 8 classes
# tweet_cluster_names = [f"Tweet Cluster {i}" for i in range(75)]
met_cluster_names = [f"Met Cluster {i}" for i in range(K)]
feature_names = met_cluster_names

global_importance = pd.Series(
    np.abs(concept_weights).mean(axis=0),
    index=feature_names
).sort_values(ascending=False)

# OR max absolute weight (a feature is important if it matters for ANY class)
global_importance_max = pd.Series(
    np.abs(concept_weights).max(axis=0),
    index=feature_names
).sort_values(ascending=False)

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_weight': np.abs(concept_weights).mean(axis=0),
    'max_abs_weight': np.abs(concept_weights).max(axis=0)
}).sort_values('mean_abs_weight', ascending=False)
important_csv_path = f"{env_vars['RESULTS_DIR']}/cluster_lr/{config.dataset_out_name}/polarity/{CLUSTER_TEC}/{EMB_NAME}/k_{K}_importance.csv"
if CLUSTER_TEC == "pckmeans":
    important_csv_path = important_csv_path.replace("_importance.csv", f"_w{PCK_WEIGHT}_importance.csv")
importance_df.to_csv(important_csv_path)
print(f"Feature importance saved to: {important_csv_path}")